# Small SAGE-like CNN Weight VAE + Post-Hoc Latent Smoothing

Cache-first experiment for testing paper-like post-hoc decoder latent smoothing on a small VAE trained over TinyCNN weights. The post-hoc NF is trained only on frozen decoder geometry, never on downstream task loss or preconditioner targets.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.sage_cnn_vae_smoothing import (
    fast_config,
    paperish_config,
    run_or_load,
)

plt.rcParams['figure.dpi'] = 130


## Hardcoded Config

`FAST_CONFIG` is the default. Switch `ACTIVE_CONFIG = PAPERISH_CONFIG` for a heavier run. All artifacts are written under `artifacts/loss_landscape_analysis/sage_cnn_vae_smoothing/{run_label}`.


In [ ]:
FAST_CONFIG = fast_config(
    run_label='sage_cnn_vae_smoothing_fast',
    dataset_name='fashion_mnist',
    download=True,
    cache_first=True,
    force_rerun=False,
)

PAPERISH_CONFIG = paperish_config(
    run_label='sage_cnn_vae_smoothing_paperish',
    dataset_name='fashion_mnist',
    download=True,
    cache_first=True,
    force_rerun=False,
)

ACTIVE_CONFIG = FAST_CONFIG
ACTIVE_CONFIG


## Run Or Load


In [ ]:
tables = run_or_load(ACTIVE_CONFIG)
output_dir = Path(tables.output_dir)
print('output_dir:', output_dir)
print('vae_metrics:', tables.vae_metrics.shape)
print('geometry:', tables.geometry.shape)
print('selected_lrs:', tables.selected_lrs.shape)
print('downstream_results:', tables.downstream_results.shape)
print('downstream_curves:', tables.downstream_curves.shape)


## VAE Reconstruction Quality


In [ ]:
vae_quality = tables.vae_metrics[tables.vae_metrics.get('record_type', '') == 'vae_quality'].copy()
vae_history = tables.vae_metrics[tables.vae_metrics.get('record_type', '') == 'vae_train_history'].copy()

if not vae_quality.empty:
    display(vae_quality[[
        'reconstruction_mse',
        'reconstruction_rel_l2',
        'latent_roundtrip_l2',
        'raw_test_acc',
        'decoded_test_acc',
        'raw_test_loss',
        'decoded_test_loss',
    ]].describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
if not vae_history.empty:
    axes[0].plot(vae_history['step'], vae_history['train_recon_mse'], label='train recon')
    axes[0].plot(vae_history['step'], vae_history['val_recon_mse'], label='val recon')
    axes[0].set_yscale('log')
    axes[0].legend()
if not vae_quality.empty:
    axes[1].scatter(vae_quality['raw_test_acc'], vae_quality['decoded_test_acc'])
    lo = min(float(vae_quality['raw_test_acc'].min()), float(vae_quality['decoded_test_acc'].min()))
    hi = max(float(vae_quality['raw_test_acc'].max()), float(vae_quality['decoded_test_acc'].max()))
    axes[1].plot([lo, hi], [lo, hi], color='black', linewidth=1)
axes[0].set_title('VAE normalized reconstruction MSE')
axes[0].set_xlabel('step')
axes[0].grid(True, alpha=0.25)
axes[1].set_title('decoded vs raw test accuracy')
axes[1].set_xlabel('raw')
axes[1].set_ylabel('decoded')
axes[1].grid(True, alpha=0.25)
path = output_dir / 'vae_reconstruction_quality.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Post-Hoc Geometry


In [ ]:
display(tables.geometry)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
geom = tables.geometry.set_index('coordinate')
geom['isometry_objective'].plot(kind='bar', ax=axes[0])
geom['condition_median'].plot(kind='bar', ax=axes[1])
geom['log_eig_spread_median'].plot(kind='bar', ax=axes[2])
axes[0].set_title('relaxed isometry objective')
axes[1].set_title('median pullback condition')
axes[2].set_title('median log eig spread')
for ax in axes:
    ax.grid(True, axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=35)
path = output_dir / 'posthoc_geometry.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Learning Rates And Downstream Metrics


In [ ]:
selected = tables.selected_lrs.copy()
display(selected.sort_values(['method', 'selected'], ascending=[True, False]))

results = tables.downstream_results.copy()
summary = results.groupby('method', as_index=False).agg(
    median_aulc=('aulc', 'median'),
    median_final_train_loss=('final_train_loss', 'median'),
    median_best_train_loss=('best_train_loss', 'median'),
    median_final_test_loss=('final_test_loss', 'median'),
    median_final_test_acc=('final_test_acc', 'median'),
    median_reconstruction_rel_l2=('reconstruction_rel_l2', 'median'),
    diverged_rate=('diverged', 'mean'),
)
display(summary.sort_values('median_aulc'))


## Downstream Curves


In [ ]:
curves = tables.downstream_curves.copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for method, rows in curves.groupby('method'):
    train_pivot = rows.pivot(index='step', columns='start_index', values='train_loss')
    acc_pivot = rows.pivot(index='step', columns='start_index', values='test_acc')
    axes[0].plot(train_pivot.index, train_pivot.median(axis=1), label=method)
    axes[1].plot(acc_pivot.index, acc_pivot.median(axis=1), label=method)
axes[0].set_yscale('log')
axes[0].set_title('median train loss')
axes[1].set_title('median test accuracy')
for ax in axes:
    ax.set_xlabel('step')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
path = output_dir / 'downstream_curves.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Paired AULC And Random-NF Control


In [ ]:
pivot = results.pivot_table(index='start_index', columns='method', values='aulc', aggfunc='median')
display(pivot)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
if {'decoder_latent', 'decoder_trained_nf'}.issubset(pivot.columns):
    axes[0].scatter(pivot['decoder_latent'], pivot['decoder_trained_nf'])
    lo = float(min(pivot['decoder_latent'].min(), pivot['decoder_trained_nf'].min()))
    hi = float(max(pivot['decoder_latent'].max(), pivot['decoder_trained_nf'].max()))
    axes[0].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[0].set_xlabel('decoder latent AULC')
    axes[0].set_ylabel('trained NF AULC')
if {'decoder_random_nf', 'decoder_trained_nf'}.issubset(pivot.columns):
    axes[1].scatter(pivot['decoder_random_nf'], pivot['decoder_trained_nf'])
    lo = float(min(pivot['decoder_random_nf'].min(), pivot['decoder_trained_nf'].min()))
    hi = float(max(pivot['decoder_random_nf'].max(), pivot['decoder_trained_nf'].max()))
    axes[1].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[1].set_xlabel('random NF AULC')
    axes[1].set_ylabel('trained NF AULC')
for ax in axes:
    ax.set_title('lower is better')
    ax.grid(True, alpha=0.25)
path = output_dir / 'paired_aulc.png'
fig.savefig(path, bbox_inches='tight')
print('saved:', path)


## Interpretation


In [ ]:
display(Markdown(tables.interpretation_markdown))
print((output_dir / 'interpretation.md').read_text(encoding='utf-8'))


## Files Written


In [ ]:
for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(path)
